In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yasserh/student-marks-dataset/Student_Marks.csv


In [2]:
train=pd.read_csv("/kaggle/input/datasets/yasserh/student-marks-dataset/Student_Marks.csv")

In [3]:
print(train.shape)
print(train.columns)
print(train.isna().sum())
print(train.dtypes)

(100, 3)
Index(['number_courses', 'time_study', 'Marks'], dtype='object')
number_courses    0
time_study        0
Marks             0
dtype: int64
number_courses      int64
time_study        float64
Marks             float64
dtype: object


In [4]:
X=train.drop(['Marks'],axis=1)
y=train['Marks']

In [5]:
y.shape

(100,)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score

linear_pipeline=Pipeline(
    steps=[
        ("linear",LinearRegression())        
    ]
)
ridge_pipeline=Pipeline(
    steps=[
        ("ridge",Ridge(alpha=0.1))
    ]
)
lasso_pipeline=Pipeline(
    steps=[
        ("lasso",Lasso(alpha=0.1))
    ]
)

linear_pipeline.fit(X_train, y_train)
ridge_pipeline.fit(X_train, y_train)
lasso_pipeline.fit(X_train,y_train)

Pipeline(steps=[('lasso', Lasso(alpha=0.1))])

In [8]:
linear_predict=linear_pipeline.predict(X_test)
ridge_predict=ridge_pipeline.predict(X_test)
lasso_predict=lasso_pipeline.predict(X_test)
print(linear_predict)
print(ridge_predict)
print(lasso_predict)

[39.26916601 40.04420284 18.51258697 41.05167011 15.75171287 46.37721326
  5.37135287  7.24988902 28.18484492 22.87471178  7.86609996  0.23833218
 42.51111707 36.73945794 31.93075879 39.88411254 31.13329447 37.51161823
 39.77312882 39.85835016]
[39.26559859 40.04047962 18.51364457 41.0477393  15.75335186 46.37206202
  5.37528794  7.25347309 28.18376422 22.8747974   7.86954672  0.24342788
 42.50686107 36.7364542  31.92884343 39.88044189 31.13148921 37.50839175
 39.76941532 39.85465146]
[39.20844209 39.95969075 18.5067686  40.94275503 15.79667366 46.29768704
  5.46550994  7.25212486 28.13170335 22.85735467  7.86670579  0.32432915
 42.39834145 36.68542563 31.86770849 39.7782859  31.15930573 37.52075748
 39.75454787 39.79606772]


In [9]:
print("y:", y.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("linear_predict:", linear_predict.shape)

y: (100,)
y_train: (80,)
y_test: (20,)
X_train: (80, 2)
X_test: (20, 2)
linear_predict: (20,)


In [10]:
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

print("Linear Regression")
print(mean_absolute_error(y_test, linear_predict))
print(mean_squared_error(y_test,linear_predict))
print(r2_score(y_test,linear_predict))

print("----")
print("Ridge ")
print(mean_absolute_error(y_test, ridge_predict))
print(mean_squared_error(y_test,ridge_predict))
print(r2_score(y_test,ridge_predict))

print("----")
print("Lasso ")
print(mean_absolute_error(y_test, lasso_predict))
print(mean_squared_error(y_test,lasso_predict))
print(r2_score(y_test,lasso_predict))

Linear Regression
2.772215988953804
12.290775858513403
0.9363318018156712
----
Ridge 
2.771350599634734
12.28855389150282
0.936343311962599
----
Lasso 
2.7561388461879117
12.208472934408006
0.9367581442161359


| Model  | Alpha | MAE_mean | MAE_std | MSE_mean | MSE_std | Notes         |
| ------ | ----- | -------- | ------- | -------- | ------- | ------------- |
| Linear | –     | 3.193    | 0.195   | 13.489   | 1.712   | baseline      |
| Ridge  | 1.0   | 3.193    | 0.205   | 13.477   | 1.751   | default alpha |
| Lasso  | 1.0   | 3.244    | 0.211   | 13.871   | 1.916   | default alpha |

| Model  | Alpha | MAE_mean | MAE_std | MSE_mean | MSE_std |
| ------ | ----- | -------- | ------- | -------- | ------- |
| Linear | –     | 3.041    | 0.210   | 12.521   | 1.857   |
| Ridge  | 0.1   | 3.041    | 0.209   | 12.520   | 1.854   |
| Lasso  | 0.1   | 3.045    | 0.203   | 12.519   | 1.823   |

In [11]:

mae_scores = cross_val_score(
    estimator=linear_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)
mae_scores = np.abs(mae_scores)
MAE_mean = mae_scores.mean()
MAE_std = mae_scores.std()

mse_scores = cross_val_score(
    estimator=linear_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)
mse_scores = np.abs(mse_scores)

MSE_mean = mse_scores.mean()

MSE_std = mse_scores.std()

In [12]:
print("mae_scores: ",mae_scores)
print("MAE_mean: ",MAE_mean)
print("MAE_std: ",MAE_std)

print("mse_scores: ",mse_scores)
print("MSE_mean: ",MSE_mean)
print("MSE_std: ",MSE_std)

mae_scores:  [3.51972741 2.83303549 3.48910494 3.3107442  2.65893731]
MAE_mean:  3.1623098713333486
MAE_std:  0.35167616352372766
mse_scores:  [14.97819555 11.93196662 14.60933607 14.3358768   9.79014592]
MSE_mean:  13.129104193583988
MSE_std:  1.982337312443021


In [13]:

mae_scores = cross_val_score(
    estimator=ridge_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)
mae_scores = np.abs(mae_scores)
MAE_mean = mae_scores.mean()
MAE_std = mae_scores.std()

mse_scores = cross_val_score(
    estimator=ridge_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)
mse_scores = np.abs(mse_scores)

MSE_mean = mse_scores.mean()

MSE_std = mse_scores.std()

In [14]:
print("mae_scores: ",mae_scores)
print("MAE_mean: ",MAE_mean)
print("MAE_std: ",MAE_std)

print("mse_scores: ",mse_scores)
print("MSE_mean: ",MSE_mean)
print("MSE_std: ",MSE_std)

mae_scores:  [3.51904595 2.83273532 3.49092316 3.31032861 2.65966071]
MAE_mean:  3.1625387511999064
MAE_std:  0.351690822653123
mse_scores:  [14.96684263 11.93305546 14.62597528 14.32783094  9.79320493]
MSE_mean:  13.129381850414507
MSE_std:  1.9805861500958644


In [15]:

mae_scores = cross_val_score(
    estimator=lasso_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)
mae_scores = np.abs(mae_scores)
MAE_mean = mae_scores.mean()
MAE_std = mae_scores.std()

mse_scores = cross_val_score(
    estimator=lasso_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)
mse_scores = np.abs(mse_scores)

MSE_mean = mse_scores.mean()

MSE_std = mse_scores.std()

In [16]:
print("mae_scores: ",mae_scores)
print("MAE_mean: ",MAE_mean)
print("MAE_std: ",MAE_std)

print("mse_scores: ",mse_scores)
print("MSE_mean: ",MSE_mean)
print("MSE_std: ",MSE_std)

mae_scores:  [3.51067403 2.82108676 3.52939705 3.29925173 2.66330589]
MAE_mean:  3.1647430927726696
MAE_std:  0.3578469588369543
mse_scores:  [14.84848773 11.91084737 14.95000724 14.24441898  9.79534686]
MSE_mean:  13.149821635739906
MSE_std:  2.005584022858997
